# 6. Models C/D/E: writer-dependent SVM verification

For each unseen test writer:

* positives = their first `n_ref` genuine signatures (the "specimens on file")
* negatives = genuine signatures of the **training** writers ("random forgeries"); validation writers are kept
  out so they can pick the threshold
* an RBF SVM on CNN embeddings scores the writer's remaining genuine signatures and all 24 skilled forgeries.
  **No skilled forgery is used to build the verifier**, which matches a bank that only has genuine specimens.

Reports AUC / EER over all test writers, accuracy / FAR / FRR at one global threshold chosen on the validation
writers, and the mean per-writer EER (each writer gets their own threshold, the usual writer-dependent figure).

Feature extractors (`ckpt`): `imagenet` (C, no fine-tuning), `binary` (model A's features), `writerid` (D) and
`writeridF` (E) from `writer_id.ipynb`. `mode = "skel"` uses stroke-normalised input (leak check).
Results go to `runs/wdsvm/<ckpt>_<mode>_ref<n>.json`; existing ones are loaded unless `RERUN = True`.

In [1]:
EXPERIMENTS = [            # (ckpt, input mode, references per writer)
    ("imagenet", "binary", 5),
    ("imagenet", "skel", 5),
    ("binary", "binary", 5),
    ("writerid", "binary", 5),
    ("writerid", "skel", 5),
    ("writeridF", "binary", 5),
    ("writerid", "binary", 1),
    ("writerid", "binary", 3),
    ("writerid", "binary", 10),
]
FOLDS = [0, 1, 2, 3, 4]
RERUN = False

In [2]:
%run ./common.ipynb

In [3]:
import json

KEYS = ["auc", "eer", "acc", "far", "frr", "mean_writer_eer"]


def run_wd_fold(F, meta, fold, n_ref):
    """Negatives = genuine signatures of the training writers only. The global threshold is
    the EER point on the validation writers, applied unchanged to the test writers."""
    _, tr, va, te = writer_split(fold=fold)
    neg = F[tr[meta.label.values[tr] == 0]]
    yv, sv, _ = score_writers(F, meta, sorted(set(meta.writer[va])), neg, n_ref)
    thr, _ = eer_threshold(yv, sv)
    yt, st, per_writer = score_writers(F, meta, sorted(set(meta.writer[te])), neg, n_ref)
    return {"fold": fold, **metrics(yt, st, thr), "mean_writer_eer": float(np.mean(per_writer))}


def run_experiment(tag, mode, n_ref):
    path = os.path.join(RUNS, "wdsvm", f"{tag}_{mode}_ref{n_ref}.json")
    if os.path.exists(path) and not RERUN:
        return json.load(open(path))
    meta, *_ = writer_split()
    rows = []
    for k in FOLDS:
        r = run_wd_fold(cnn_features(tag, mode, k), meta, k, n_ref)
        rows.append(r)
        print(f"  [{tag}, {mode}, {n_ref} refs] fold {k}: AUC {r['auc']:.4f}  EER {r['eer']:.4f}  "
              f"acc {r['acc']:.4f}  FAR {r['far']:.4f}  FRR {r['frr']:.4f}", flush=True)
    res = {"features": tag, "mode": mode, "n_ref": n_ref, "folds": rows,
           **{f"{k}_mean": float(np.mean([r[k] for r in rows])) for k in KEYS},
           **{f"{k}_std": float(np.std([r[k] for r in rows])) for k in KEYS}}
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(res, f, indent=2)
    return res

In [4]:
table = []
for tag, mode, n_ref in EXPERIMENTS:
    r = run_experiment(tag, mode, n_ref)
    table.append({"features": tag, "input": mode, "refs": n_ref,
                  **{k: f"{r[k + '_mean']:.3f} +/- {r[k + '_std']:.3f}" for k in KEYS}})
pd.DataFrame(table)

,features,input,refs,auc,eer,acc,far,frr,mean_writer_eer
0,imagenet,binary,5,0.903 +/- 0.024,0.176 +/- 0.027,0.821 +/- 0.034,0.201 +/- 0.063,0.151 +/- 0.026,0.132 +/- 0.018
1,imagenet,skel,5,0.879 +/- 0.021,0.196 +/- 0.028,0.798 +/- 0.030,0.177 +/- 0.025,0.233 +/- 0.045,0.153 +/- 0.016
2,binary,binary,5,0.860 +/- 0.048,0.216 +/- 0.052,0.756 +/- 0.074,0.305 +/- 0.161,0.166 +/- 0.069,0.167 +/- 0.029
3,writerid,binary,5,0.931 +/- 0.018,0.138 +/- 0.029,0.844 +/- 0.045,0.208 +/- 0.097,0.091 +/- 0.026,0.099 +/- 0.016
4,writerid,skel,5,0.923 +/- 0.018,0.153 +/- 0.025,0.830 +/- 0.043,0.217 +/- 0.097,0.110 +/- 0.031,0.123 +/- 0.027
5,writeridF,binary,5,0.911 +/- 0.024,0.171 +/- 0.034,0.807 +/- 0.043,0.241 +/- 0.097,0.132 +/- 0.034,0.111 +/- 0.020
6,writerid,binary,1,0.870 +/- 0.043,0.213 +/- 0.045,0.776 +/- 0.046,0.299 +/- 0.144,0.145 +/- 0.080,0.140 +/- 0.043
7,writerid,binary,3,0.922 +/- 0.013,0.155 +/- 0.019,0.834 +/- 0.044,0.231 +/- 0.111,0.091 +/- 0.050,0.109 +/- 0.017
8,writerid,binary,10,0.944 +/- 0.016,0.130 +/- 0.039,0.860 +/- 0.032,0.172 +/- 0.052,0.084 +/- 0.020,0.100 +/- 0.029
